In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import os

In [2]:
def metrics(tn, fp, fn, tp):
    """
    Calculate confusion matrix and various performance metrics.
    Args:
        tn : True Negatives
        fp : False Positives
        fn : False Negatives
        tp : True Positives
    Returns:
        cm_df (pd.DataFrame): Confusion matrix as a DataFrame.
        metric_df (pd.DataFrame): Performance metrics as a DataFrame.
    """
    tn = tn.astype(np.float32)
    fp = fp.astype(np.float32)
    fn = fn.astype(np.float32)
    tp = tp.astype(np.float32)

    cm_dict = {'predict\\actual':['Positive', 'Negative']
               ,'Positive':[tp, fn]
               ,'Negative':[fp, tn]}

    cm_df = pd.DataFrame(cm_dict) #數值排成表格，方便之後存檔或印出來看
    
    accuracy = round((tp + tn) / (tp + tn + fp + fn), 4) if (tp + tn + fp + fn) > 0 else 0  
    sensitivity = round(tp / (tp + fn), 4) if (tp + fn) > 0 else 0
    specificity = round(tn / (tn + fp), 4) if (tn + fp) > 0 else 0
    ppv = round(tp / (tp + fp), 4) if (tp + fp) > 0 else 0
    npv = round(tn / (tn + fn), 4) if (tn + fn) > 0 else 0
    mcc = round((tp * tn - fp * fn) / np.sqrt(float(tp + fp) * float(tp + fn) * float(tn + fp) * float(tn + fn)), 4) if (tp + fp) > 0 and (tp + fn) > 0 and (tn + fp) > 0 and (tn + fn) > 0 else 0

    metric_dict = {'metric':['Accuruacy', 'Sensitivity', 'Specificity', 'PPV', 'NPV', 'MCC'],
                   'Value':[accuracy, sensitivity, specificity, ppv, npv, mcc]}

    metric_df = pd.DataFrame(metric_dict)

    return cm_df, metric_df


def train_majority_vote(y_label, yhat_proba, set_name='train'):
    """
    Evaluate binary classification performance at a specific threshold.
    Args:
        y_true_label (pd.DataFrame): True/predict binary labels (0 or 1).
        y_pred_proba (pd.DataFrame): Predicted probabilities for the positive class.
        threshold (float): Threshold to convert probabilities to binary predictions.
    Returns:
        result_df (pd.DataFrame): DataFrame with research_id, true labels, and predicted labels.
        cm_df (pd.DataFrame): Confusion matrix as a DataFrame.
        metric_df (pd.DataFrame): Performance mestrics as a DataFrame.
    """
    y_label = y_label.rename(columns={'file_name': 'research_id'})
    yhat_proba = yhat_proba.rename(columns={'file_name': 'research_id', f'yhat_{set_name}_original':f'yhat_{set_name}_proba'})
    
    research_id = y_label['research_id'].to_numpy()
    y_true_label = y_label[f'y_{set_name}'].to_numpy() #樣本為單位  
    y_pred_proba = yhat_proba[f'yhat_{set_name}_proba'].to_numpy()

    unique_ids, id_indices, id_counts = np.unique(research_id, return_inverse=True, return_counts=True)

    y_true_label_subject = (np.bincount(id_indices, weights=y_true_label) / id_counts).astype(int)  #weights:加種值 bincount:標籤加起來
    #把同一個人的標籤加總，再除以視窗數 確保都是0/1
    if len(np.unique(y_true_label_subject)) != 2:
        raise ValueError('Subject-level true labels must contain both classes 0 and 1 for meaningful evaluation.')

    proba_threshold_range = np.sort(np.unique(np.round(y_pred_proba*10000)))/10000 # Generate thresholds based on unique predicted probabilities
    #所有出現過的預測機率排序

    best_proba_threshold = None
    best_label_threshold = None
    best_cm = (0, 0, 0, 0)
    best_MCC = -np.inf

    for ii, current_proba_threshold in enumerate(tqdm(proba_threshold_range)): #:enumerate在單獨數值清單+索引
        # if ii > 2:
        #     break
        binary_preds = (y_pred_proba >= current_proba_threshold).astype(int) #單看每個視窗的機率 以樣本為單位

        yhat_score_subject = np.bincount(id_indices, weights=binary_preds) / id_counts #計算每個病人「得票率」  看0/1誰多

        label_threshold_range = np.sort(np.unique(yhat_score_subject))

        for jj, current_label_threshold in enumerate(label_threshold_range):

            y_pred_final = (yhat_score_subject >= current_label_threshold).astype(int) #得票率 0.75 >= 門檻 0.5，則判為病人(1)

            tp = np.sum((y_true_label_subject == 1) & (y_pred_final == 1))
            tn = np.sum((y_true_label_subject == 0) & (y_pred_final == 0))
            fp = np.sum((y_true_label_subject == 0) & (y_pred_final == 1))
            fn = np.sum((y_true_label_subject == 1) & (y_pred_final == 0))
            MCC_value = round((tp * tn - fp * fn) / np.sqrt(float(tp + fp) * float(tp + fn) * float(tn + fp) * float(tn + fn)), 4) if (tp + fp) > 0 and (tp + fn) > 0 and (tn + fp) > 0 and (tn + fn) > 0 else 0

            if MCC_value > best_MCC:
                best_MCC = MCC_value
                best_label_threshold = float(current_label_threshold)
                best_proba_threshold = float(current_proba_threshold)
                best_cm = (tn, fp, fn, tp)
    
    threshold_df = pd.DataFrame({'threshold':['best_proba_threshold', 'best_label_threshold'],
                                 'value':[best_proba_threshold, best_label_threshold]})
    best_cm_df, best_metric_df = metrics(best_cm[0], best_cm[1], best_cm[2], best_cm[3])



    return best_cm_df, best_metric_df, best_proba_threshold, best_label_threshold, threshold_df


def test_majority_vote(y_label, yhat_proba, set_name='test', best_proba_threshold=0.5, best_label_threshold=0.5):
    """
    Evaluate binary classification performance at a specific threshold.
    Args:
        y_true_label (pd.DataFrame): True/predict binary labels (0 or 1).
        y_pred_proba (pd.DataFrame): Predicted probabilities for the positive class.
        threshold (float): Threshold to convert probabilities to binary predictions.
    Returns:
        result_df (pd.DataFrame): DataFrame with research_id, true labels, and predicted labels.
        cm_df (pd.DataFrame): Confusion matrix as a DataFrame.
        metric_df (pd.DataFrame): Performance mestrics as a DataFrame.
    """
    y_label = y_label.rename(columns={f'file_name': 'research_id'})
    yhat_proba = yhat_proba.rename(columns={'file_name': 'research_id', f'yhat_{set_name}_original':f'yhat_{set_name}_proba'})
    
    research_id = y_label['research_id'].to_numpy()
    y_true_label = y_label[f'y_{set_name}'].to_numpy()
    y_pred_proba = yhat_proba[f'yhat_{set_name}_proba'].to_numpy()

    unique_ids, id_indices, id_counts = np.unique(research_id, return_inverse=True, return_counts=True)

    y_true_label_subject = (np.bincount(id_indices, weights=y_true_label) / id_counts).astype(int)
    if len(np.unique(y_true_label_subject)) != 2:
        raise ValueError('Subject-level true labels must contain both classes 0 and 1 for meaningful evaluation.')

    binary_preds = (y_pred_proba >= best_proba_threshold).astype(int)

    yhat_score_subject = np.bincount(id_indices, weights=binary_preds) / id_counts #依照 ID 把票數加總算出比例 以人為單位

    y_pred_final = (yhat_score_subject >= best_label_threshold).astype(int)

    tp = np.sum((y_true_label_subject == 1) & (y_pred_final == 1))
    tn = np.sum((y_true_label_subject == 0) & (y_pred_final == 0))
    fp = np.sum((y_true_label_subject == 0) & (y_pred_final == 1))
    fn = np.sum((y_true_label_subject == 1) & (y_pred_final == 0))

    best_cm_df, best_metric_df = metrics(tn, fp, fn, tp)

    return best_cm_df, best_metric_df



In [3]:
data_dir = r'D:\M143020071\xgboost_result\static_dynamic_ECGFounder_askna_mr_feature_result\combine\多數決前處理/'
data_combine_dir=r'D:\M143020071\xgboost_result\static_dynamic_ECGFounder_askna_mr_feature_result\combine'
save_dir = os.path.join(data_combine_dir, '多數決/')

if not os.path.exists(save_dir):
    os.makedirs(save_dir)
    print(f'Created directory: {save_dir}')
else:
    print(f'Directory already exists: {save_dir}')

Directory already exists: D:\M143020071\xgboost_result\static_dynamic_ECGFounder_askna_mr_feature_result\combine\多數決/


In [4]:
y_train_label = pd.read_csv(os.path.join(data_combine_dir, 'train_label.csv'))
y_train_probability = pd.read_csv(os.path.join(data_combine_dir, 'train_prob.csv'))
y_test_label = pd.read_csv(os.path.join(data_combine_dir, 'test_label.csv'))
y_test_probability = pd.read_csv(os.path.join(data_combine_dir, 'test_prob.csv'))

In [5]:
train_best_cm_df, train_best_metric_df, train_best_proba_threshold, train_best_label_threshold, train_threshold_df = train_majority_vote(y_train_label, y_train_probability, 'train')
print(train_best_cm_df)
print('\n')
print(train_best_metric_df)
print('\n')
print(train_threshold_df)
print('\n')
train_best_cm_df.to_csv(save_dir + 'train_cm.csv', index=False)
train_best_metric_df.to_csv(save_dir + 'train_metric.csv', index=False)
train_threshold_df.to_csv(save_dir + 'train_thresholds.csv', index=False)

test_best_cm_df, test_best_metric_df = test_majority_vote(y_test_label, y_test_probability, 'test', train_best_proba_threshold, train_best_label_threshold)
print(test_best_cm_df)
print('\n')
print(test_best_metric_df) 
test_best_cm_df.to_csv(save_dir + 'test_cm.csv', index=False)
test_best_metric_df.to_csv(save_dir + 'test_metric.csv', index=False)

100%|██████████| 1803/1803 [00:10<00:00, 178.95it/s]

  predict\actual  Positive  Negative
0       Positive      90.0       0.0
1       Negative       0.0     432.0


        metric  Value
0    Accuruacy    1.0
1  Sensitivity    1.0
2  Specificity    1.0
3          PPV    1.0
4          NPV    1.0
5          MCC    1.0


              threshold   value
0  best_proba_threshold  0.0045
1  best_label_threshold  1.0000


  predict\actual  Positive  Negative
0       Positive      29.0     168.0
1       Negative      61.0     264.0


        metric   Value
0    Accuruacy  0.5613
1  Sensitivity  0.3222
2  Specificity  0.6111
3          PPV  0.1472
4          NPV  0.8123
5          MCC -0.0520
